<a href="https://colab.research.google.com/github/Eliezer-Carvalho/IP/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **[Hugging Face](https://huggingface.co/)**

Plataforma padrão na indústria para modelos de Inteligência Artificial. <br>
Foi a plataforma usada para o download dos modelos. <br>

O código seguinte foi rodado tanto em CPU como em GPU.

## Carregar Modelos

In [ ]:
#### Carregar Modelos ####
### Duas Opções -> Upload Local ou Upload via Hugging Face
### Como o objetivo é hospedar um modelo local foi feito o download do modelo para o ambiente pessoal para começar desde cedo a estabelecer algumas boas práticas.
### Este processo acarreta custos de memória e no caso de treino na GPU (via Google Colab) é dispendioso em termos de tempo dar upload do modelo no ecossistema Google.

#Libraries
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODELO_NAME = "x" #Se for via HF usar o nome do modelo
MODELO_PATH = "x" #Se for via ambiente pessoal é meter o caminho que indica a pasta

modelo = AutoModelForCausalLM.from_pretrained (MODELO_PATH, device_map = "auto") #device_map pode ser "cpu" ou "cuda"

tokenizer = AutoTokenizer.from_pretrained (MODELO_PATH) #load do tokenizer do modelo, importante para prompt processing e token generation

## Inferência dos Modelos

In [ ]:
#### Inferência ####
#### A Inferência pode ser um processo bem mais trabalhado (top_k, top_p, etc) mas aqui está uma versão simples.

input = tokenizer (prompt, return_tensor = "pt") #return_tensor = pytorch

probs = modelo (**input)

tokens = modelo.generate (**input, max_new_tokens = x) #** desfaz dict

texto = tokenizer.decode (tokens[0]) #token a token

# **[llama.cpp](https://github.com/ggml-org/llama.cpp)**

É uma ferramenta que permite realizar inferência de maneira prática e eficiente na CPU. <br>
Em CPU llama.cpp é o que domina.

In [ ]:
#### llama.cpp não é como uma biblioteca em Python, é mais uma ferramenta que permite executar comandos no terminal ####
#### Alguns dos comandos importantes:

convert_hf_to_gguf.py	-> Comando que converte o formato de ficheiro para GGUF (formato de ficheiro excelente para correr modelos na CPU)

llama-bench -> Permite realizar métricas de benchmark no modelo e descobrir o valor de Prompt Processing/s e Token Generation/s

llama-quantize -> Permite aplicar Quantização a um modelo

llama-cli -> Chat Mode (-c, -t, -b, --temp, --top_k, --top_p, -n)

llama-server -> Lança um servidor estilo OpenAI

# **Quantização**

Quantização tem como objetivo converter o tipo de dados do modelo para um tipo mais pequeno em termos de memória. <br>
Existe uma ferramenta para CPU e outra para GPU.

## CPU

In [ ]:
llama-quantize -> Permite aplicar Quantização a um modelo

## GPU

In [ ]:
#### Foi utilizada a biblioteca BitsandBytes da Hugging Face ####

from transformers import BitsAndBytesConfig

quantization = BitsAndBytesConfig (
    load_in_4bit = True,  #Quantização
    bnb_4bit_quant_type = 'nf4',  #Tipo de Quantização
    bnb_4bit_use_double_quant = True, #Double Quantization
    bnb_4bit_compute_dtype = 'float16', #Tipo de precisão usada nos cálculos durante a inferência.
)

modelo = AutoModelForCausalLM.from_pretrained (MODELO_PATH, device_map = 'auto', quantization_config = quantization)

tokenizer = AutoTokenizer.from_pretrained (MODELO_PATH)

modelo.save_pretrained ('/Modelos')
tokenizer.save_pretrained ('/Modelos')

# **CPU vs CPU (llama.cpp) vs GPU**

In [ ]:
import torch
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu' #GPU -> CUDA | CPU -> normal

modelo = AutoModelForCausalLM.from_pretrained (MODELO_PATH).to(device)

tokenizer = AutoTokenizer.from_pretrained (MODELO_PATH)

for i in range (10):
  inputs = tokenizer (prompt, return_tensors = 'pt').to(device)
  #print (inputs['inputs_id'].shape()) #Para ver número de tokens processados

  ####### Prompt Processing
  torch.cuda.synchronize() #Só se usa se for CUDA, sicroniza com a GPU
  start_pp = time.time()

  with torch.no_grad():
    probs = MODEL (**inputs) #Prompt Processing

  torch.cuda.synchronize() #Só se usa se for CUDA, sicroniza com a GPU
  end_pp = time.time()
  ################################


  ######## Token Generation
  torch.cuda.synchronize() #Só se usa se for CUDA, sicroniza com a GPU
  start_tp = time.time()

  output_tokens = MODEL.generate (**inputs, max_new_tokens = 100) #Token Generation

  torch.cuda.synchronize() #Só se usa se for CUDA, sicroniza com a GPU
  end_tp = time.time()
  #####################################

  text = tokenizer.decode (output_tokens [0])
  print (text)

  elapsed_time_pp = end_pp - start_pp #tempo total = terminou - começou
  elapsed_time_tp = end_tp - start_tp

  return elapsed_time_pp, elapsed_time_tp

In [ ]:
Em llama.cpp é só correr o comando llama-bench